# MP 5 — Fault-Tolerance Reconciliation

**Abdulla Mansoor (CIS 731) · abdulla03@ksu.edu**

This notebook is the evidence for the *Fault-Tolerance Reconciliation* section
of `report.md`. It re-reads the artifacts produced by `./run_all.sh` and checks
them; it does not recompute them, so nothing here can quietly disagree with the
submitted numbers.

The run it inspects was hard-killed with `os._exit(1)` after 10 committed
micro-batches and then restarted from `checkpointLocation`.

## 1. What the pipeline was asked to do

Watermark delay is not a guess. `analyze_lateness.py` measures the lateness
distribution `ingest_ts - ts_end` over the admitted records and rounds p99 up
to the next 30-second boundary.

In [1]:
import json, pandas as pd
from pathlib import Path
pd.set_option("display.width", 120)

lat = json.loads(Path("../out/lateness.json").read_text())
man = json.loads(Path("../landing_manifest.json").read_text())
print("fixture:", man["records_total"], "records, seed", man["seed"],
      "| file partition order:", man["file_partition_order"])
print()
print("lateness (seconds):")
for k, v in lat["lateness_sec"].items():
    print(f"  {k:>7} {v:>9.3f}")
print()
print("watermark delay:", lat["watermark_delay_sec"], "s  rule:", lat["watermark_rule"])
print("records beyond that delay:", lat["records_beyond_watermark"],
      f"({lat['completeness_at_watermark']:.3%} completeness)")

fixture: 1000 records, seed 20261020 | file partition order: arrival

lateness (seconds):
      p50     4.147
      p95    26.098
      p99   160.591
      min     0.212
      max   592.900
     mean    10.758
   stddev    34.591

watermark delay: 180 s  rule: ceil(p99 / 30s) * 30s
records beyond that delay: 8 (99.152% completeness)


## 2. The contracts actually rejected something

A quarantine path that is never exercised is an assertion, not a gate. The
fixture injects a known population of violations; the counts below are the
contracts firing on them. Records violating more than one contract appear
under each, which is why the per-contract counts sum to more than the
record count.

In [2]:
print("admitted  ", lat["records_admitted"])
print("quarantined", lat["records_quarantined"],
      f"({lat['quarantine_rate']:.1%})")
print()
print("injected by the generator (ground truth):")
for k, v in sorted(man["violations_by_kind"].items(), key=lambda x: -x[1]):
    print(f"  {k:<20} {v:>3}")
print()
print("contracts that fired:")
for k, v in lat["violations_by_contract"].items():
    print(f"  {k:<20} {v:>3}")
print()
print("sum of contract hits:", sum(lat["violations_by_contract"].values()),
      "across", lat["records_quarantined"], "records")

admitted   943
quarantined 57 (5.7%)

injected by the generator (ground truth):
  unlisted_source       13
  overlong_chunk        13
  unknown_license        8
  blank_source           8
  blank_text             8
  reversed_interval      7

contracts that fired:
  SourceWhitelist       21
  BoundedChunking       20
  IngestCausality       12
  NonEmptyText           8
  Provenance             8
  LicenseGate            8
  TemporalOrdering       7

sum of contract hits: 84 across 57 records


## 3. Restart continuity

The checkpoint commit log — not `metrics.jsonl` — is authoritative. A commit
is written by the engine before a batch is acknowledged; the metrics file is
written by the driver and can lose its last rows to a hard kill. Both are
shown so the difference is visible.

In [3]:
rec = json.loads(Path("../out/reconciliation.json").read_text())
rows = []
for q, c in rec["restart_continuity"].items():
    rows.append({
        "query": q,
        "committed": len(c["committed_batches"]),
        "last_batch_before_kill": max(c["metrics_pre_kill"]) if c["metrics_pre_kill"] else None,
        "first_batch_after_restart": min(c["metrics_post_restart"]) if c["metrics_post_restart"] else None,
        "reprocessed": len(c["reprocessed"]),
        "progress_rows_lost_to_kill": c["progress_rows_lost_to_kill"],
    })
display(pd.DataFrame(rows))
print("\nreprocessed batches across all queries:",
      sum(len(c["reprocessed"]) for c in rec["restart_continuity"].values()))

,query,committed,last_batch_before_kill,first_batch_after_restart,reprocessed,progress_rows_lost_to_kill
0,qa_tumbling_60s,63,9,10,0,[]
1,qb_tokens_60s,61,8,9,0,[]
2,qc_sliding_5m_1m,63,9,10,0,[]
3,quarantine,40,8,10,0,[9]



reprocessed batches across all queries: 0


## 4. No duplicate window keys

This is the property the restart has to preserve. Append mode plus a
watermark means a window is written once, when it closes; if the restart had
replayed committed batches, a window key would appear twice.

In [4]:
d = rec["duplicate_check"]
display(pd.DataFrame([
    {"sink": "qa_tumbling_60s", "rows": d["qa_rows"], "distinct_keys": d["qa_distinct_keys"], "key": "window_start"},
    {"sink": "qb_tokens_60s",   "rows": d["qb_rows"], "distinct_keys": d["qb_distinct_keys"], "key": "window_start"},
    {"sink": "qc_sliding_5m_1m","rows": d["qc_rows"], "distinct_keys": d["qc_distinct_keys"], "key": "(window_start, token)"},
    {"sink": "quarantine",      "rows": d["quarantine_rows"], "distinct_keys": d["quarantine_distinct_keys"], "key": "record_id"},
]))

,sink,rows,distinct_keys,key
0,qa_tumbling_60s,46,46,window_start
1,qb_tokens_60s,46,46,window_start
2,qc_sliding_5m_1m,1655,1655,"(window_start, token)"
3,quarantine,57,57,record_id


## 5. Every record is accounted for

The reconciliation is a closed balance over all 1000 records, not a spot
check. A record is emitted in a closed window, discarded by the watermark,
still held in a window that had not closed when the replay drained, or
quarantined by a contract. Nothing else is possible, and the four buckets
have to sum to the input.

In [5]:
a = rec["accounting"]
print(f"  emitted in closed windows      {a['emitted_in_closed_windows']:>5}")
print(f"  dropped behind the watermark    {a['dropped_behind_watermark_per_engine']:>5}   (engine counter)")
print(f"  held in windows open at EOS     {a['held_in_windows_open_at_eos']:>5}   ({a['windows_still_open']} windows)")
print(f"  quarantined by a contract       {a['quarantined']:>5}")
print(f"  {'-' * 38}")
print(f"  total                          {a['sum']:>5}   input {a['records_total']}")
assert a["sum"] == a["records_total"], "accounting does not balance"
print()
print(f"windows in batch truth {a['windows_in_batch_truth']},",
      f"emitted {a['windows_emitted']}, still open {a['windows_still_open']}")
print("final watermark:", a["final_watermark"])
print("completeness of the emitted windows:",
      f"{a['completeness_of_emitted_windows']:.3%}")

  emitted in closed windows        875
  dropped behind the watermark        4   (engine counter)
  held in windows open at EOS        64   (4 windows)
  quarantined by a contract          57
  --------------------------------------
  total                           1000   input 1000

windows in batch truth 50, emitted 46, still open 4
final watermark: 2026-10-19T14:46:15.000Z
completeness of the emitted windows: 99.545%


### 5b. Where the prediction and the engine disagree

The watermark model in `reconcile.py` reproduces the engine's reported
watermark for all 40 non-empty batches, and predicts four of the five drops
the engine performed. The fifth record sits on the late-data filter boundary.
It is reported rather than tuned away.

In [6]:
p = rec["independent_prediction"]
print("predicted closed windows", p["predicted_closed_windows"],
      "| engine", a["windows_emitted"])
print("predicted drops", p["predicted_drop_count"],
      "| engine", p["engine_drop_count"],
      "| residual", p["residual_records"], "record")
print("watermark model agreement:", p["watermark_model_batches_matched"])
print()
display(pd.DataFrame(p["predicted_dropped"]))

predicted closed windows 46 | engine 46
predicted drops 5 | engine 4 | residual 1 record
watermark model agreement: 40 of 40 non-empty QA batches



,record_id,behind_watermark_sec
0,c000101,353.0
1,c000232,161.0
2,c000306,30.0
3,c000426,141.0
4,c000555,37.0


## 6. Query output

QA is the 60-second tumbling record-grain aggregate; QC is the 5-minute /
1-minute sliding token aggregate. The top-10 is computed here, at read time,
rather than in the stream — an in-stream top-K would have to hold every
candidate term for every open window and then sort, and a rank over a
streaming aggregate is not supported in append mode.

In [7]:
import glob
qa = pd.concat([pd.read_parquet(f) for f in glob.glob("../out/qa_tumbling_60s/*.parquet")])
qa = qa.sort_values("window_start").reset_index(drop=True)
print(f"QA: {len(qa)} windows, {qa.record_count.sum()} records")
display(qa.head(8))

QA: 46 windows, 875 records


,window_start,window_end,record_count,mean_ingest_latency_sec,approx_distinct_sources,min_ts_end,max_ts_end
0,2026-10-19 14:00:00,2026-10-19 14:01:00,16,8.2626,16,2026-10-19 14:00:19,2026-10-19 14:00:57
1,2026-10-19 14:01:00,2026-10-19 14:02:00,21,13.2502,20,2026-10-19 14:01:01,2026-10-19 14:01:56
2,2026-10-19 14:02:00,2026-10-19 14:03:00,18,4.7189,16,2026-10-19 14:02:00,2026-10-19 14:02:59
3,2026-10-19 14:03:00,2026-10-19 14:04:00,17,8.7377,17,2026-10-19 14:03:08,2026-10-19 14:03:59
4,2026-10-19 14:04:00,2026-10-19 14:05:00,16,10.8018,16,2026-10-19 14:04:02,2026-10-19 14:04:57
5,2026-10-19 14:05:00,2026-10-19 14:06:00,18,5.1634,18,2026-10-19 14:05:01,2026-10-19 14:05:51
6,2026-10-19 14:06:00,2026-10-19 14:07:00,20,19.3882,18,2026-10-19 14:06:09,2026-10-19 14:06:55
7,2026-10-19 14:07:00,2026-10-19 14:08:00,14,5.3856,14,2026-10-19 14:07:03,2026-10-19 14:07:51


In [8]:
qc = pd.concat([pd.read_parquet(f) for f in glob.glob("../out/qc_sliding_5m_1m/*.parquet")])
top = (qc.groupby("token", as_index=False)["term_count"].sum()
         .sort_values(["term_count", "token"], ascending=[False, True]).head(10))
top.insert(0, "rank", range(1, len(top) + 1))
print("Top-10 terms, summed over the sliding windows:")
display(top.reset_index(drop=True))
print("distinct tokens:", qc.token.nunique(),
      "| distinct sliding windows:", qc.window_start.nunique(),
      "| (window, token) rows:", len(qc))

Top-10 terms, summed over the sliding windows:


,rank,token,term_count
0,1,trap,13656
1,2,drill,7306
2,3,boom,5381
3,4,bap,4017
4,5,funk,3376
5,6,hyphy,2852
6,7,crunk,2707
7,8,memphis,2236
8,9,atlanta,2102
9,10,compton,1898


distinct tokens: 36 | distinct sliding windows: 46 | (window, token) rows: 1655


## 7. State growth stayed inside the predicted bound

For a tumbling window the number of open window keys is bounded by
`ceil((watermark + window) / window)` plus the batch in flight and the
one-batch watermark lag. For the sliding window the same count is multiplied
by the live vocabulary. These are bounds, not measurements — the point is
that the observed maximum does not exceed them.

In [9]:
ms = json.loads(Path("../out/metrics_summary.json").read_text())
import math
WM, VOCAB = lat["watermark_delay_sec"], qc.token.nunique()
qa_bound = math.ceil((WM + 60) / 60) + 2
qc_bound = (math.ceil((WM + 300) / 60) + 2) * VOCAB
rows = []
for q, v in ms["queries"].items():
    rows.append({
        "query": q,
        "state_rows_median": v["stateOperators.numRowsTotal"]["median"],
        "state_rows_max": v["stateOperators.numRowsTotal"]["max"],
        "predicted_bound": qa_bound if "60s" in q else (qc_bound if "sliding" in q else 0),
        "rows_dropped_by_watermark": v["rows_dropped_by_watermark"],
    })
display(pd.DataFrame(rows))
print(f"\nQA/QB bound = ceil(({WM}+60)/60) + 2 = {qa_bound} window keys")
print(f"QC bound    = (ceil(({WM}+300)/60) + 2) x {VOCAB} tokens = {qc_bound} rows")

,query,state_rows_median,state_rows_max,predicted_bound,rows_dropped_by_watermark
0,qa_tumbling_60s,5.0,6.0,6,4
1,qb_tokens_60s,5.0,6.0,6,4
2,qc_sliding_5m_1m,315.0,352.0,360,102
3,quarantine,NaN,NaN,0,0



QA/QB bound = ceil((180+60)/60) + 2 = 6 window keys
QC bound    = (ceil((180+300)/60) + 2) x 36 tokens = 360 rows


## 8. Throughput, latency and the pinned environment

Throughput and latency are liveness properties (INV-5), read off this log
rather than asserted by a contract. `batchDuration` is
`durationMs.triggerExecution`; there is no top-level `batchDuration` field on
`StreamingQueryProgress`, and reading one returns `None`.

The figures are low in absolute terms because the replay is deliberately rate
limited to one file every three seconds — this measures the pipeline, not the
machine.

In [10]:
rows = []
for q, v in ms["queries"].items():
    rows.append({
        "query": q,
        "batches": v["batches_logged"],
        "with_input": v["batches_with_input"],
        "inputRows/s (mean)": v["inputRowsPerSecond"]["mean"],
        "processedRows/s (mean)": v["processedRowsPerSecond"]["mean"],
        "batchDuration ms (median)": v["batchDuration_ms"]["median"],
        "batchDuration ms (p95)": v["batchDuration_ms"]["p95"],
    })
display(pd.DataFrame(rows))
e = ms["environment"]
print()
for k in ("python", "pyspark", "java", "platform", "accelerator"):
    print(f"  {k:<12} {e[k]}")
print(f"  {'source':<12} sha256 {e['source_sha256']}")
print(f"  {'deps':<12} sha256 {e['pip_freeze_sha256']} (pip freeze)")

,query,batches,with_input,inputRows/s (mean),processedRows/s (mean),batchDuration ms (median),batchDuration ms (p95)
0,qa_tumbling_60s,64,40,29.80,15.04,1667,3013
1,qb_tokens_60s,62,40,27.13,15.10,1633,6422
2,qc_sliding_5m_1m,64,40,27.23,14.44,1773,5420
3,quarantine,40,39,20.92,21.28,1146,4405



  python       3.13.16
  pyspark      3.5.1
  java         openjdk version "21.0.12.1" 2026-08-18
  platform     Linux-6.18.44-fc-v139-x86_64-with-glibc2.39
  accelerator  none (CPU baseline, INV-14/INV-15)
  source       sha256 8e701cd7e6256f6c7e6e91d5f17879bd885154f75ee7ac8052e69ee945fd7206
  deps         sha256 40df0f6da681ac9b68096b1daefde471e3ea6064d6c912cf250f5c80c3986d9c (pip freeze)


## 9. Verdict

In [11]:
print("RECONCILIATION:", rec["result"])
if rec["failures"]:
    for f in rec["failures"]:
        print("  -", f)
else:
    print("  all invariants hold; no duplicate or missing window keys,")
    print("  no batch reprocessed across the restart, and the record")
    print("  accounting balances exactly against the input.")

RECONCILIATION: PASS
  all invariants hold; no duplicate or missing window keys,
  no batch reprocessed across the restart, and the record
  accounting balances exactly against the input.
